# CycleGAN Vergleichs-Inference (mehrere Runs)

Wählt die neuesten CycleGAN-Experimente (oder explizite Checkpoints), zieht 5 Zufallsbilder aus einem Eingabe-Ordner, generiert alle Outputs, speichert sie unter `results/cyclegan_inference/<exp>/<direction>` und zeigt Eingabe + alle Ausgaben nebeneinander. Die GAN-Outputs werden auf die Originalgröße des Eingangs zurückskaliert (keine feste 16:9-Streckung). Die Anzahl der Spalten richtet sich nach der Anzahl der Checkpoints.


In [5]:
from pathlib import Path
import os, random, torch
from PIL import Image
from matplotlib import pyplot as plt

os.chdir("/srv/store/docker-users/thesis/khajuria/day2night")
print("Working dir:", os.getcwd())

# Parameter anpassen
INPUT_DIR = Path("data/Straßen Bilder/test")
NUM_SAMPLES = 12
DIRECTION = "day2night"  # oder "night2day"
#CKPT_PATHS = ["experiments/cyclegan_day2night_20251215_142123/epoch_0200.pt", "experiments/cyclegan_day2night_20251217_003630/epoch_0200.pt"]
CKPT_PATHS = ["experiments/cyclegan_day2night_20251218_015621/epoch_0200.pt", "experiments/cyclegan_day2night_20251218_051158/epoch_0200.pt", "experiments/cyclegan_day2night_20251218_092009/epoch_0200.pt"]
#CKPT_PATHS = ["experiments/cyclegan_day2night_20251216_232145/epoch_0200.pt", "experiments/cyclegan_day2night_20251215_142123/epoch_0200.pt","experiments/cyclegan_day2night_20251212_023200/epoch_0200.pt", "experiments/cyclegan_day2night_20251218_005055/epoch_0200.pt", "experiments/cyclegan_day2night_20251216_180414/epoch_0200.pt" ]  # optional: Liste expliziter Checkpoint-Pfade; leer => neueste Experimente
NUM_LATEST = 2    # wie viele der neuesten Experimente nehmen, falls CKPT_PATHS leer ist
BASE_CONFIG = Path("configs/cyclegan.yaml")
OUTPUT_ROOT = Path("results/cyclegan_inference_demo")
SEED = 6

random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda:7" if torch.cuda.is_available() else "cpu")


Working dir: /srv/store/docker-users/thesis/khajuria/day2night


In [6]:
from src.utils.common import load_cfg
from src.apply_cyclegan import (
    build_inference_transform,
    build_generator,
    tensor_to_pil,
    gather_image_paths,
)

def load_state_dict(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(path, map_location="cpu")

def latest_ckpt(exp_dir: Path) -> Path:
    candidates = sorted(exp_dir.glob("epoch_*.pt"))
    if not candidates:
        raise FileNotFoundError(f"Keine epoch_*.pt in {exp_dir}")
    return candidates[-1]

def pick_recent_experiments(root: Path, n: int):
    exps = [p for p in root.glob("cyclegan_*") if p.is_dir()]
    if len(exps) < n:
        raise RuntimeError(f"Zu wenige Experimente in {root}; gefunden {len(exps)}")
    exps.sort(key=lambda p: p.stat().st_mtime)
    return exps[-n:]

def resolve_ckpts() -> list[Path]:
    if CKPT_PATHS:
        return [Path(p) for p in CKPT_PATHS]
    exps = pick_recent_experiments(Path("experiments"), n=NUM_LATEST)
    return [latest_ckpt(exp) for exp in exps]

def cfg_for_ckpt(ckpt: Path):
    local_cfg = ckpt.parent / "cyclegan.yaml"
    if local_cfg.is_file():
        return load_cfg(local_cfg)
    return load_cfg(BASE_CONFIG)

ckpts = resolve_ckpts()
print("Checkpoints (Reihenfolge = Spaltenreihenfolge):")
for c in ckpts:
    print(" -", c)

base_cfg = load_cfg(BASE_CONFIG)
extensions = base_cfg.get("data", {}).get("extensions", ("jpg", "jpeg", "png", "bmp", "tif", "tiff"))
image_paths = gather_image_paths(INPUT_DIR, extensions)
assert image_paths, f"Keine Bilder in {INPUT_DIR}"
sample_paths = random.sample(image_paths, min(NUM_SAMPLES, len(image_paths)))
print(f"Verwende {len(sample_paths)} Zufallsbilder aus {INPUT_DIR}")


Checkpoints (Reihenfolge = Spaltenreihenfolge):
 - experiments/cyclegan_day2night_20251218_015621/epoch_0200.pt
 - experiments/cyclegan_day2night_20251218_051158/epoch_0200.pt
 - experiments/cyclegan_day2night_20251218_092009/epoch_0200.pt
Verwende 12 Zufallsbilder aus data/Straßen Bilder/test


In [7]:
def run_generator(ckpt: Path):
    cfg = cfg_for_ckpt(ckpt)
    transform = build_inference_transform(cfg)
    generator = build_generator(cfg, device, DIRECTION)
    state = load_state_dict(ckpt)
    key = "G" if DIRECTION == "day2night" else "F"
    generator.load_state_dict(state[key])
    generator.eval()
    out_dir = OUTPUT_ROOT / ckpt.parent.name / DIRECTION
    out_dir.mkdir(parents=True, exist_ok=True)

    saved = {}  # str(input_path) -> output_path
    brightness_gain = float(cfg.get("inference", {}).get("brightness_gain", 1.0))
    output_gamma = float(cfg.get("inference", {}).get("output_gamma", 1.0))

    with torch.inference_mode():
        for path in sample_paths:
            img = Image.open(path).convert("RGB")
            inp = transform(img).unsqueeze(0).to(device)
            out = generator(inp)
            pil = tensor_to_pil(out, brightness_gain=brightness_gain, output_gamma=output_gamma)
            pil = pil.resize(img.size, Image.BICUBIC)  # zurück auf Originalgröße
            dest = out_dir / path.relative_to(INPUT_DIR)
            dest.parent.mkdir(parents=True, exist_ok=True)
            pil.save(dest)
            saved[str(path)] = dest
    print(f"Gespeichert unter {out_dir}")
    return saved

results = {}  # ckpt -> {input_path_str: output_path}
for ckpt in ckpts:
    results[str(ckpt)] = run_generator(ckpt)


Building day2night generator | encoder: experiments/seg_20251204_174637/encoder_GE.pth | freeze=True
Gespeichert unter results/cyclegan_inference_demo/cyclegan_day2night_20251218_015621/day2night
Building day2night generator | encoder: experiments/seg_20251204_174637/encoder_GE.pth | freeze=True
Gespeichert unter results/cyclegan_inference_demo/cyclegan_day2night_20251218_051158/day2night
Building day2night generator | encoder: experiments/seg_20251204_174637/encoder_GE.pth | freeze=True
Gespeichert unter results/cyclegan_inference_demo/cyclegan_day2night_20251218_092009/day2night


In [ ]:
# Visualisierung: Eingabe + eine Spalte pro Checkpoint
ckpt_keys = list(results.keys())  # behält Reihenfolge der ausgewählten Checkpoints
rows = len(sample_paths)
cols = 1 + len(ckpt_keys)
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
if rows == 1:
    axes = axes.reshape(1, cols)

for row, img_path in enumerate(sample_paths):
    imgs = [(Image.open(img_path), f"Eingabe\n{img_path.name}")]
    for ckpt_key in ckpt_keys:
        out_path = results[ckpt_key][str(img_path)]
        title = Path(ckpt_key).parent.name
        imgs.append((Image.open(out_path), title))

    for col, (im, title) in enumerate(imgs):
        ax = axes[row, col]
        ax.imshow(im)
        ax.set_title(title)
        ax.axis("off")

plt.tight_layout()
plt.show()
